# Data Preparation for Whisper Fine-tuning
Converts custom audio datasets from Google Drive into the HuggingFace dataset format required by the Whisper fine-tuning pipeline.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Dependencies

In [ ]:
!pip install -q datasets soundfile librosa

## Step 3: Configure Paths

In [ ]:
from pathlib import Path

# Source data folders in Google Drive
TRAIN_DATA_DIR = Path("/content/drive/MyDrive/train_25hrs_wav_flat_colab")
TEST_DATA_DIR  = Path("/content/drive/MyDrive/test_5hrs_wav_flat")

# Output saved back to Drive under 27Jun2026/whisper/
OUTPUT_BASE    = Path("/content/drive/MyDrive/27Jun2026/whisper")
TRAIN_OUT_DIR  = OUTPUT_BASE / "train_dataset"
TEST_OUT_DIR   = OUTPUT_BASE / "test_dataset"

# Temporary manifest files stored locally in Colab runtime
WORK_DIR       = Path("/content/whisper_prep")
TRAIN_PREP_DIR = WORK_DIR / "train"
TEST_PREP_DIR  = WORK_DIR / "test"

for d in [TRAIN_PREP_DIR, TEST_PREP_DIR, TRAIN_OUT_DIR, TEST_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Source folders:")
print(f"  Train : {TRAIN_DATA_DIR}  (exists: {TRAIN_DATA_DIR.exists()})")
print(f"  Test  : {TEST_DATA_DIR}   (exists: {TEST_DATA_DIR.exists()})")
print("\nOutput folders (Drive):")
print(f"  Train : {TRAIN_OUT_DIR}")
print(f"  Test  : {TEST_OUT_DIR}")

## Step 4: Explore Data Structure

In [ ]:
def explore_folder(folder: Path, label: str):
    wav_files = sorted(folder.glob("*.wav"))
    txt_files = sorted(folder.glob("*.txt"))
    print(f"{label}: {folder}")
    print(f"  .wav files : {len(wav_files)}")
    print(f"  .txt files : {len(txt_files)}")
    if wav_files:
        print(f"  Sample wav : {wav_files[0].name}")
    if txt_files:
        print(f"  Sample txt : {txt_files[0].name}")
    return wav_files, txt_files

train_wavs, train_txts = explore_folder(TRAIN_DATA_DIR, "TRAIN")
print()
test_wavs,  test_txts  = explore_folder(TEST_DATA_DIR,  "TEST")

## Step 5: Create Missing Transcription `.txt` Files (if needed)

Each `.wav` must have a paired `.txt` with the same filename stem containing its transcription.  
If any are missing, empty placeholders are created — **fill them with actual transcriptions before running Step 7**.

In [ ]:
def check_and_create_txt_files(wav_files, folder: Path, label: str):
    missing = []
    for wav in wav_files:
        txt = folder / (wav.stem + ".txt")
        if not txt.exists():
            txt.write_text("", encoding="utf-8")
            missing.append(txt.name)

    if missing:
        print(f"[{label}] Created {len(missing)} empty .txt placeholder(s) in {folder}")
        print(f"  Please add transcriptions before continuing.")
        print(f"  First 5: {missing[:5]}")
    else:
        print(f"[{label}] All .wav files already have paired .txt transcription files. Ready to proceed.")

check_and_create_txt_files(train_wavs, TRAIN_DATA_DIR, "TRAIN")
check_and_create_txt_files(test_wavs,  TEST_DATA_DIR,  "TEST")

## Step 6: Build `audio_paths` and `text` Manifest Files

Required format for `data_prep.py`:
```
audio_paths →  <utt_id>  <absolute_path_to_audio>
text        →  <utt_id>  <transcription>
```
Files with empty transcriptions are skipped with a warning.

In [ ]:
def build_manifests(wav_files, data_dir: Path, prep_dir: Path, split: str):
    audio_lines = []
    text_lines  = []
    skipped     = []

    for idx, wav in enumerate(sorted(wav_files), start=1):
        txt = data_dir / (wav.stem + ".txt")
        transcript = txt.read_text(encoding="utf-8").strip() if txt.exists() else ""
        if not transcript:
            skipped.append(wav.name)
            continue
        utt_id = f"{split}_{idx:05d}"
        audio_lines.append(f"{utt_id} {wav.resolve()}")
        text_lines.append(f"{utt_id} {transcript}")

    (prep_dir / "audio_paths").write_text("\n".join(audio_lines), encoding="utf-8")
    (prep_dir / "text").write_text("\n".join(text_lines), encoding="utf-8")

    print(f"[{split.upper()}] Manifest written: {len(audio_lines)} entries → {prep_dir}")
    if skipped:
        print(f"  WARNING: {len(skipped)} file(s) skipped (empty transcription): {skipped[:5]}")

build_manifests(train_wavs, TRAIN_DATA_DIR, TRAIN_PREP_DIR, "train")
build_manifests(test_wavs,  TEST_DATA_DIR,  TEST_PREP_DIR,  "test")

## Step 7: Verify Manifest Files

In [ ]:
def verify_manifests(prep_dir: Path, label: str):
    audio_lines = (prep_dir / "audio_paths").read_text(encoding="utf-8").strip().splitlines()
    text_lines  = (prep_dir / "text").read_text(encoding="utf-8").strip().splitlines()
    match = len(audio_lines) == len(text_lines)
    print(f"{label}")
    print(f"  audio_paths entries : {len(audio_lines)}")
    print(f"  text entries        : {len(text_lines)}")
    print(f"  Counts match        : {match}")
    if not match:
        print("  ERROR: entry counts differ — check for missing transcriptions!")
    else:
        print(f"  Sample audio : {audio_lines[0]}")
        print(f"  Sample text  : {text_lines[0]}")
    return match

train_ok = verify_manifests(TRAIN_PREP_DIR, "TRAIN")
print()
test_ok  = verify_manifests(TEST_PREP_DIR,  "TEST")

## Step 8: Convert to HuggingFace Dataset Format and Save to Drive

In [ ]:
from datasets import Dataset, Audio, Value

def run_data_prep(prep_dir: Path, output_dir: Path, label: str, ok: bool):
    if not ok:
        print(f"[{label}] Skipping — manifest verification failed.")
        return None

    scp_entries = (prep_dir / "audio_paths").read_text(encoding="utf-8").strip().splitlines()
    txt_entries = (prep_dir / "text").read_text(encoding="utf-8").strip().splitlines()

    audio_paths    = [line.split(maxsplit=1)[1].strip() for line in scp_entries]
    transcriptions = [" ".join(line.split()[1:]).strip() for line in txt_entries]

    dataset = Dataset.from_dict({"audio": audio_paths, "sentence": transcriptions})
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
    dataset = dataset.cast_column("sentence", Value("string"))
    dataset.save_to_disk(str(output_dir))

    print(f"[{label}] Saved {len(dataset)} samples → {output_dir}")
    print(f"  Features: {dataset.features}")
    return dataset

train_dataset = run_data_prep(TRAIN_PREP_DIR, TRAIN_OUT_DIR, "TRAIN", train_ok)
test_dataset  = run_data_prep(TEST_PREP_DIR,  TEST_OUT_DIR,  "TEST",  test_ok)

## Step 9: Sanity Check — Listen to a Sample

In [ ]:
import IPython.display as ipd

if train_dataset:
    sample = train_dataset[0]
    print("Train sample:")
    print(f"  Transcription : {sample['sentence']}")
    print(f"  Sampling rate : {sample['audio']['sampling_rate']} Hz")
    print(f"  Duration      : {len(sample['audio']['array']) / sample['audio']['sampling_rate']:.2f} sec")
    ipd.display(ipd.Audio(sample['audio']['array'], rate=sample['audio']['sampling_rate']))

## Step 10: Summary

In [ ]:
print("Data Preparation Complete")
print("=" * 50)
if train_dataset:
    print(f"Train : {TRAIN_OUT_DIR}  ({len(train_dataset)} samples)")
if test_dataset:
    print(f"Test  : {TEST_OUT_DIR}  ({len(test_dataset)} samples)")
print()
print("Pass these to the fine-tuning script:")
print(f"  --train_datasets {TRAIN_OUT_DIR}")
print(f"  --eval_datasets  {TEST_OUT_DIR}")